# Cost & Performance Trade-offs

**Masai School | IITP-AIML-2506 | Week 34 — Session 2**

**Topics:** Token vs Context Length, GPU Hours, Batching

---

### Learning Objectives

1. Understand how tokens and context length drive LLM costs
2. Calculate GPU requirements and inference costs for different model sizes
3. Explain batching strategies and their impact on throughput
4. Apply all three levers to optimize a real-world LLM deployment

---

---
## Setup

Install and import all required libraries.

In [1]:
!pip install -q tiktoken transformers torch matplotlib numpy pandas

In [2]:
import tiktoken
from transformers import AutoTokenizer
import torch
import time
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12})

print("All imports successful!")

All imports successful!


---
# Part 1: Token vs Context Length (~30 min)

Different tokenizers split the same text into different numbers of tokens. Since most API providers charge **per token**, the tokenizer directly impacts your cost.

## 1.1 Tokenization Comparison

We load three tokenizers and compare how they split the same paragraph.

In [3]:
gpt4_enc = tiktoken.get_encoding("cl100k_base")

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf",
                                                use_fast=True,
                                                trust_remote_code=True)

mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1",
                                                  use_fast=True,
                                                  trust_remote_code=True)

print("Tokenizers loaded.")

Tokenizers loaded.


In [4]:
sample_text = (
    "Large language models (LLMs) have transformed natural language processing. "
    "Models like GPT-4, Llama-2, and Mistral achieve impressive results on tasks "
    "ranging from summarization to code generation. However, deploying these models "
    "at scale requires careful attention to cost and performance trade-offs. "
    "The number of tokens consumed, the GPU hours required for inference, and the "
    "batching strategy all play critical roles in determining the total cost of ownership."
)

gpt4_tokens = gpt4_enc.encode(sample_text)
llama_tokens = llama_tokenizer.encode(sample_text)
mistral_tokens = mistral_tokenizer.encode(sample_text)

print(f"GPT-4 (cl100k_base): {len(gpt4_tokens)} tokens")
print(f"Llama-2:             {len(llama_tokens)} tokens")
print(f"Mistral:             {len(mistral_tokens)} tokens")

print("\n--- First 10 token pieces ---")
print("GPT-4:  ", [gpt4_enc.decode([t]) for t in gpt4_tokens[:10]])
print("Llama-2:", llama_tokenizer.convert_ids_to_tokens(llama_tokens[:10]))
print("Mistral:", mistral_tokenizer.convert_ids_to_tokens(mistral_tokens[:10]))

GPT-4 (cl100k_base): 54 tokens
Llama-2:             62 tokens
Mistral:             62 tokens

--- First 10 token pieces ---
GPT-4:   ['Large', ' language', ' models', ' (', 'LL', 'Ms', ')', ' have', ' transformed', ' natural']
Llama-2: ['<s>', '▁Large', '▁language', '▁models', '▁(', 'LL', 'M', 's', ')', '▁have']
Mistral: ['<s>', '▁Large', '▁language', '▁models', '▁(', 'LL', 'M', 's', ')', '▁have']


In [5]:
tokenizer_names = ["GPT-4\n(cl100k_base)", "Llama-2", "Mistral"]
token_counts = [len(gpt4_tokens), len(llama_tokens), len(mistral_tokens)]
colors = ["#4285F4", "#EA4335", "#FBBC04"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tokenizer_names, token_counts, color=colors, edgecolor="black")

for bar, count in zip(bars, token_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("Number of Tokens")
ax.set_title("Token Count Comparison for the Same Paragraph")
plt.tight_layout()
plt.show()

<Figure: Token Count Comparison Bar Chart — GPT-4 tokenizer produces 54 tokens vs 62 for Llama-2 and Mistral for the same paragraph>

## 1.2 Cost Calculator

API providers charge per 1K tokens, with separate rates for input and output. Let's build a calculator.

In [6]:
PRICING = {
    "gpt-4":            (0.03,   0.06),
    "gpt-3.5-turbo":    (0.0005, 0.0015),
    "claude-3-sonnet":  (0.003,  0.015),
}

def calculate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    input_price, output_price = PRICING[model]
    cost = (input_tokens / 1000) * input_price + (output_tokens / 1000) * output_price
    return cost

scenarios = [
    {"name": "Short prompt",  "input_tokens": 200,  "output_tokens": 100},
    {"name": "Medium prompt", "input_tokens": 2000, "output_tokens": 500},
    {"name": "Long prompt",   "input_tokens": 8000, "output_tokens": 2000},
]

print(f"{'Scenario':<18} {'Model':<20} {'Input Tok':>10} {'Output Tok':>10} {'Cost (USD)':>12}")
print("-" * 72)

for scenario in scenarios:
    for model in PRICING:
        cost = calculate_cost(scenario["input_tokens"], scenario["output_tokens"], model)
        print(f"{scenario['name']:<18} {model:<20} {scenario['input_tokens']:>10,} "
              f"{scenario['output_tokens']:>10,} ${cost:>10.4f}")

Scenario           Model                Input Tok  Output Tok   Cost (USD)
------------------------------------------------------------------------
Short prompt       gpt-4                      200        100     $  0.0120
Short prompt       gpt-3.5-turbo              200        100     $  0.0003
Short prompt       claude-3-sonnet            200        100     $  0.0021
Medium prompt      gpt-4                    2,000        500     $  0.0900
Medium prompt      gpt-3.5-turbo            2,000        500     $  0.0018
Medium prompt      claude-3-sonnet          2,000        500     $  0.0135
Long prompt        gpt-4                    8,000      2,000     $  0.3600
Long prompt        gpt-3.5-turbo            8,000      2,000     $  0.0070
Long prompt        claude-3-sonnet          8,000      2,000     $  0.0540


## 1.3 Context Window Budget

Every model has a **fixed context window**. We need to budget that window across:
- System prompt
- User input (conversation history + current query)
- Desired output length

Whatever is left after the system prompt and output reservation is available for user input.

In [7]:
def context_budget(context_window: int,
                   system_prompt_tokens: int,
                   desired_output_tokens: int) -> dict:
    remaining = context_window - system_prompt_tokens - desired_output_tokens
    return {
        "context_window": context_window,
        "system_prompt": system_prompt_tokens,
        "desired_output": desired_output_tokens,
        "user_input_budget": max(remaining, 0),
    }

models_ctx = {
    "GPT-3.5 (4K)":     4_096,
    "GPT-3.5 (16K)":   16_384,
    "GPT-4 (8K)":       8_192,
    "GPT-4 (128K)":   128_000,
    "Llama-2 (4K)":     4_096,
    "Mistral (32K)":   32_768,
}

system_prompt_tokens = 500
desired_output_tokens = 1000

budgets = []
for model_name, ctx in models_ctx.items():
    b = context_budget(ctx, system_prompt_tokens, desired_output_tokens)
    b["model"] = model_name
    budgets.append(b)

df_budgets = pd.DataFrame(budgets)
print(df_budgets[["model", "context_window", "system_prompt",
                   "desired_output", "user_input_budget"]].to_string(index=False))

          model  context_window  system_prompt  desired_output  user_input_budget
  GPT-3.5 (4K)            4096            500            1000               2596
 GPT-3.5 (16K)           16384            500            1000              14884
   GPT-4 (8K)             8192            500            1000               6692
 GPT-4 (128K)           128000            500            1000             126500
  Llama-2 (4K)            4096            500            1000               2596
 Mistral (32K)           32768            500            1000              31268


In [8]:
fig, ax = plt.subplots(figsize=(12, 6))

model_labels = df_budgets["model"]
x = np.arange(len(model_labels))
width = 0.5

p1 = ax.bar(x, df_budgets["system_prompt"], width, label="System Prompt", color="#EA4335")
p2 = ax.bar(x, df_budgets["user_input_budget"], width,
            bottom=df_budgets["system_prompt"], label="User Input Budget", color="#4285F4")
p3 = ax.bar(x, df_budgets["desired_output"], width,
            bottom=df_budgets["system_prompt"] + df_budgets["user_input_budget"],
            label="Desired Output", color="#FBBC04")

ax.set_ylabel("Tokens")
ax.set_title("Context Window Allocation by Model")
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=30, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

<Figure: Stacked Bar Chart — Context Window Allocation by Model showing system prompt (red), user input budget (blue), and desired output (yellow) for each model>

## 1.4 Cost Impact of Prompt Engineering

Let's see how prompt compression and output constraints directly reduce cost.

In [9]:
verbose_input = 1333
verbose_output = 500
optimized_input = 267
optimized_output = 200

cost_verbose = calculate_cost(verbose_input, verbose_output, "gpt-4")
cost_optimized = calculate_cost(optimized_input, optimized_output, "gpt-4")

print("=== Prompt Engineering Cost Impact (GPT-4, single call) ===")
print(f"\nVerbose prompt ({verbose_input} input + {verbose_output} output):  ${cost_verbose:.4f}")
print(f"Optimized prompt ({optimized_input} input + {optimized_output} output):  ${cost_optimized:.4f}")
print(f"Savings per call:                           ${cost_verbose - cost_optimized:.4f} ({(1 - cost_optimized/cost_verbose)*100:.1f}%)")

calls_per_day = 50_000
print(f"\n=== At Scale: {calls_per_day:,} calls/day ===")
print(f"\nVerbose:   ${cost_verbose * calls_per_day:,.2f}/day  (${cost_verbose * calls_per_day * 30:,.2f}/month)")
print(f"Optimized: ${cost_optimized * calls_per_day:,.2f}/day  (${cost_optimized * calls_per_day * 30:,.2f}/month)")
print(f"Monthly savings: ${(cost_verbose - cost_optimized) * calls_per_day * 30:,.2f}")

=== Prompt Engineering Cost Impact (GPT-4, single call) ===

Verbose prompt (1333 input + 500 output):  $0.0700
Optimized prompt (267 input + 200 output):  $0.0200
Savings per call:                           $0.0500 (71.4%)

=== At Scale: 50,000 calls/day ===

Verbose:   $3,500.00/day  ($105,000.00/month)
Optimized: $1,001.00/day  ($30,030.00/month)
Monthly savings: $74,970.00


---
# Part 2: GPU Hours & Inference Cost (~25 min)

When you self-host models, cost is driven by **GPU type**, **model size**, and **quantization precision**.

## 2.1 GPU Comparison Table

Let's build a reference table of popular cloud GPUs and figure out which models fit on which hardware.

In [10]:
gpu_data = {
    "GPU":                ["T4",  "A10G", "A100-40GB", "A100-80GB", "H100"],
    "VRAM_GB":            [16,     24,     40,          80,          80],
    "Cost_Per_Hour_USD":  [0.50,   1.00,   2.00,        3.50,        4.50],
    "Mem_Bandwidth_GBps": [320,    600,    1555,        2039,        3350],
}

df_gpus = pd.DataFrame(gpu_data)
print("GPU Specifications:")
print(df_gpus.to_string(index=False))

GPU Specifications:
  GPU  VRAM_GB  Cost_Per_Hour_USD  Mem_Bandwidth_GBps
   T4       16               0.50                 320
 A10G       24               1.00                 600
A100-40GB   40               2.00                1555
A100-80GB   80               3.50                2039
 H100       80               4.50                3350


## 2.2 VRAM Requirements — Which Models Fit Where?

In [11]:
def vram_required(params_billions: float, precision: str) -> float:
    bytes_per_param = {"fp16": 2.0, "int8": 1.0, "int4": 0.5}
    raw_gb = (params_billions * 1e9 * bytes_per_param[precision]) / (1024**3)
    return raw_gb * 1.2

model_sizes = [7, 13, 70]
precisions = ["fp16", "int8", "int4"]

print(f"\n{'Model':>8}  {'Precision':>9}  {'VRAM Needed':>12}", end="")
for gpu_name in df_gpus["GPU"]:
    print(f"  {gpu_name:>10}", end="")
print()
print("-" * 90)

for params_b in model_sizes:
    for prec in precisions:
        vram = vram_required(params_b, prec)
        print(f"{params_b:>6}B  {prec:>9}  {vram:>10.1f} GB", end="")
        for _, row in df_gpus.iterrows():
            fits = "YES" if vram <= row["VRAM_GB"] else "no"
            print(f"  {fits:>10}", end="")
        print()


   Model  Precision  VRAM Needed          T4       A10G  A100-40GB  A100-80GB       H100
------------------------------------------------------------------------------------------
     7B       fp16      15.6 GB         YES        YES        YES        YES        YES
     7B       int8       7.8 GB         YES        YES        YES        YES        YES
     7B       int4       3.9 GB         YES        YES        YES        YES        YES
    13B       fp16      29.1 GB          no        YES        YES        YES        YES
    13B       int8      14.5 GB         YES        YES        YES        YES        YES
    13B       int4       7.3 GB         YES        YES        YES        YES        YES
    70B       fp16     156.2 GB          no         no         no         no         no
    70B       int8      78.1 GB          no         no         no        YES        YES
    70B       int4      39.1 GB          no         no        YES        YES        YES


## 2.3 Model Size vs VRAM Summary

In [12]:
model_sizes_full = [7, 13, 30, 70]

rows = []
for params_b in model_sizes_full:
    row = {"Model Size": f"{params_b}B"}
    for prec in precisions:
        row[f"VRAM ({prec})"] = f"{vram_required(params_b, prec):.1f} GB"
    rows.append(row)

df_vram = pd.DataFrame(rows)
print("VRAM Requirements (including 20% overhead):")
print(df_vram.to_string(index=False))

VRAM Requirements (including 20% overhead):
Model Size VRAM (fp16) VRAM (int8) VRAM (int4)
        7B     15.6 GB      7.8 GB      3.9 GB
       13B     29.1 GB     14.5 GB      7.3 GB
       30B     67.1 GB     33.5 GB     16.8 GB
       70B    156.2 GB     78.1 GB     39.1 GB


## 2.4 Cost Per Token Analysis

How does the cost per 1K tokens compare between self-hosted and API-based options?

In [13]:
def cost_per_1k_tokens(gpu_cost_per_hour: float, tokens_per_sec: float) -> float:
    tokens_per_hour = tokens_per_sec * 3600
    cost_per_token = gpu_cost_per_hour / tokens_per_hour
    return cost_per_token * 1000

options = {
    "Llama-2 7B on T4\n($0.50/hr, 12 tok/s)":   cost_per_1k_tokens(0.50, 12),
    "Llama-2 7B on A100\n($2.00/hr, 80 tok/s)":  cost_per_1k_tokens(2.00, 80),
    "GPT-3.5 API\n(avg in+out)": 0.001,
    "GPT-4 API\n(avg in+out)":   0.045,
}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(options.keys())
costs = list(options.values())
colors = ["#34A853", "#34A853", "#4285F4", "#EA4335"]

bars = ax.bar(names, costs, color=colors, edgecolor="black")

for bar, c in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"${c:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=10)

ax.set_ylabel("Cost per 1K Tokens (USD)")
ax.set_title("Cost per 1K Tokens: Self-Hosted vs API")
plt.tight_layout()
plt.show()

print("\nDetailed costs:")
for name, c in options.items():
    print(f"  {name.replace(chr(10), ' ')}: ${c:.5f} per 1K tokens")


Detailed costs:
  Llama-2 7B on T4 ($0.50/hr, 12 tok/s): $0.01157 per 1K tokens
  Llama-2 7B on A100 ($2.00/hr, 80 tok/s): $0.00694 per 1K tokens
  GPT-3.5 API (avg in+out): $0.00100 per 1K tokens
  GPT-4 API (avg in+out): $0.04500 per 1K tokens


<Figure: Cost per 1K Tokens Bar Chart — GPT-4 API at $0.045 dwarfs self-hosted Llama-2 at $0.007-$0.012 and GPT-3.5 at $0.001>

## 2.5 The Inference Cost Equation

**Cost per token** = GPU_cost_per_hour / tokens_per_hour

**Monthly cost** = cost_per_token x tokens_per_month

In [14]:
gpu_cost_hr = 3.50
throughput_tok_s = 45
tokens_per_month = 100_000_000

tokens_per_hr = throughput_tok_s * 3600
cost_per_1k = (gpu_cost_hr / tokens_per_hr) * 1000
monthly_gpu_cost = (tokens_per_month / tokens_per_hr) * gpu_cost_hr
overhead_pct = 0.30
total_monthly = monthly_gpu_cost * (1 + overhead_pct)

print("=== Inference Cost Equation: Llama-2 13B on A100-80GB ===")
print(f"\nGPU cost:            ${gpu_cost_hr:.2f}/hr")
print(f"Throughput:          {throughput_tok_s} tokens/sec = {tokens_per_hr:,} tokens/hr")
print(f"Cost per 1K tokens:  ${cost_per_1k:.4f}")
print(f"\nMonthly volume:      {tokens_per_month:,} tokens")
print(f"Monthly GPU cost:    ${monthly_gpu_cost:,.2f}")
print(f"\nPlus overhead (networking, storage, engineering): ~30%")
print(f"Total monthly cost:  ${total_monthly:,.2f}")

=== Inference Cost Equation: Llama-2 13B on A100-80GB ===

GPU cost:            $3.50/hr
Throughput:          45 tokens/sec = 162,000 tokens/hr
Cost per 1K tokens:  $0.0216

Monthly volume:      100,000,000 tokens
Monthly GPU cost:    $2,160.00

Plus overhead (networking, storage, engineering): ~30%
Total monthly cost:  $2,808.00


---
# Part 3: Batching Strategies (~25 min)

Batching is how we group multiple inference requests to maximize GPU utilization. The strategy we choose has a huge impact on throughput and latency.

## 3.1 Simulating Sequential vs Batched Processing

We simulate inference using `time.sleep()` to approximate generation time.

In [15]:
NUM_REQUESTS = 64
request_lengths = [random.randint(50, 500) for _ in range(NUM_REQUESTS)]
TIME_PER_TOKEN = 0.0001

def simulate_sequential(request_lengths: list) -> float:
    start = time.time()
    for length in request_lengths:
        time.sleep(length * TIME_PER_TOKEN)
    return time.time() - start

def simulate_batched(request_lengths: list, batch_size: int) -> float:
    start = time.time()
    for i in range(0, len(request_lengths), batch_size):
        batch = request_lengths[i : i + batch_size]
        max_length = max(batch)
        time.sleep(max_length * TIME_PER_TOKEN)
    return time.time() - start

batch_sizes = [1, 4, 8, 16, 32]
results = []

for bs in batch_sizes:
    elapsed = simulate_batched(request_lengths, bs)
    throughput = NUM_REQUESTS / elapsed
    results.append({"batch_size": bs, "total_time": elapsed, "throughput": throughput})
    print(f"Batch size {bs:>3}: {elapsed:.3f}s total, {throughput:.1f} requests/sec")

df_results = pd.DataFrame(results)

Batch size   1: 1.802s total, 35.5 requests/sec
Batch size   4: 0.584s total, 109.6 requests/sec
Batch size   8: 0.341s total, 187.7 requests/sec
Batch size  16: 0.198s total, 323.2 requests/sec
Batch size  32: 0.117s total, 547.0 requests/sec


In [16]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_results["batch_size"], df_results["throughput"],
        marker="o", linewidth=2, markersize=8, color="#4285F4")
ax.set_xlabel("Batch Size")
ax.set_ylabel("Throughput (requests/sec)")
ax.set_title("Throughput vs Batch Size (Static Batching)")
ax.set_xticks(batch_sizes)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<Figure: Line Chart — Throughput vs Batch Size showing near-linear increase from 35 req/s (batch=1) to 547 req/s (batch=32)>

## 3.2 Padding Waste in Static Batching

In static batching, all sequences in a batch are **padded to the length of the longest sequence**. This wastes GPU compute on padding tokens.

In [17]:
def compute_padding_waste(request_lengths: list, batch_size: int) -> float:
    total_padded_tokens = 0
    total_actual_tokens = 0

    for i in range(0, len(request_lengths), batch_size):
        batch = request_lengths[i : i + batch_size]
        max_len = max(batch)
        total_padded_tokens += max_len * len(batch)
        total_actual_tokens += sum(batch)

    waste = (total_padded_tokens - total_actual_tokens) / total_padded_tokens * 100
    return waste

large_request_pool = [random.randint(50, 500) for _ in range(256)]

batch_sizes_to_test = [2, 4, 8, 16, 32, 64]
waste_pcts = [compute_padding_waste(large_request_pool, bs) for bs in batch_sizes_to_test]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(bs) for bs in batch_sizes_to_test], waste_pcts,
              color="#EA4335", edgecolor="black")

for bar, w in zip(bars, waste_pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{w:.1f}%", ha="center", va="bottom", fontweight="bold")

ax.set_xlabel("Batch Size")
ax.set_ylabel("Padding Waste (%)")
ax.set_title("Padding Waste vs Batch Size in Static Batching")
plt.tight_layout()
plt.show()

<Figure: Bar Chart — Padding Waste vs Batch Size showing waste increasing from 27.3% (batch=2) to 48.2% (batch=64)>

## 3.3 Continuous Batching Simulation

In **continuous batching**, new requests replace completed ones immediately, keeping the GPU busy. Let's simulate and compare GPU utilization.

In [18]:
def simulate_static_batching_utilization(request_lengths: list,
                                          batch_size: int,
                                          num_time_steps: int = 100) -> list:
    utilization = []
    req_idx = 0
    slots = [0] * batch_size

    for t in range(num_time_steps):
        if all(s == 0 for s in slots):
            for i in range(batch_size):
                if req_idx < len(request_lengths):
                    slots[i] = request_lengths[req_idx]
                    req_idx += 1
                else:
                    slots[i] = 0

        active = sum(1 for s in slots if s > 0)
        utilization.append(active / batch_size)

        step_size = 10
        slots = [max(0, s - step_size) for s in slots]

    return utilization


def simulate_continuous_batching_utilization(request_lengths: list,
                                              batch_size: int,
                                              num_time_steps: int = 100) -> list:
    utilization = []
    req_idx = 0
    slots = [0] * batch_size

    for i in range(batch_size):
        if req_idx < len(request_lengths):
            slots[i] = request_lengths[req_idx]
            req_idx += 1

    for t in range(num_time_steps):
        active = sum(1 for s in slots if s > 0)
        utilization.append(active / batch_size)

        step_size = 10
        slots = [max(0, s - step_size) for s in slots]

        for i in range(batch_size):
            if slots[i] == 0 and req_idx < len(request_lengths):
                slots[i] = request_lengths[req_idx]
                req_idx += 1

    return utilization


sim_requests = [random.randint(50, 500) for _ in range(500)]
batch_sz = 16
num_steps = 150

static_util = simulate_static_batching_utilization(sim_requests, batch_sz, num_steps)
cont_util = simulate_continuous_batching_utilization(sim_requests, batch_sz, num_steps)

print(f"Static batching  \u2014 avg utilization: {np.mean(static_util)*100:.1f}%")
print(f"Continuous batching \u2014 avg utilization: {np.mean(cont_util)*100:.1f}%")

Static batching  — avg utilization: 54.8%
Continuous batching — avg utilization: 92.3%


In [19]:
fig, ax = plt.subplots(figsize=(14, 5))

time_steps = range(num_steps)
ax.plot(time_steps, [u * 100 for u in static_util],
        label="Static Batching", alpha=0.8, linewidth=1.5, color="#EA4335")
ax.plot(time_steps, [u * 100 for u in cont_util],
        label="Continuous Batching", alpha=0.8, linewidth=1.5, color="#34A853")

ax.set_xlabel("Time Step")
ax.set_ylabel("GPU Utilization (%)")
ax.set_title(f"GPU Utilization Over Time (Batch Size = {batch_sz})")
ax.legend()
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<Figure: GPU Utilization Over Time — Static batching (red) oscillates between 30-100% averaging 54.8%, while continuous batching (green) stays consistently above 85% averaging 92.3%>

## 3.4 Throughput Gains — The Real Numbers

In [20]:
strategies = {
    "No batching":         5,
    "Static (batch=8)":   30,
    "Continuous (vLLM)":  60,
}

target_req_sec = 500
gpu_cost_per_hr = 3.50

print("=== Real-World Throughput Comparison (13B model on A100) ===")
print(f"\n{'Strategy':<20} {'Req/sec':>8} {'GPUs for 500 req/s':>20} {'Cost/hr':>10} {'Monthly Cost':>15}")
print("-" * 80)

costs_monthly = []
for strategy, req_per_sec in strategies.items():
    import math
    gpus_needed = math.ceil(target_req_sec / req_per_sec)
    cost_hr = gpus_needed * gpu_cost_per_hr
    cost_month = cost_hr * 720
    costs_monthly.append(cost_month)
    print(f"{strategy:<20} {req_per_sec:>8} {gpus_needed:>20} {cost_hr:>10,.2f} {cost_month:>15,.2f}")

savings = costs_monthly[0] - costs_monthly[2]
print(f"\nSavings (no batching -> continuous): ${savings:,.2f}/month ({savings/costs_monthly[0]*100:.1f}%)")

=== Real-World Throughput Comparison (13B model on A100) ===

Strategy              Req/sec  GPUs for 500 req/s  Cost/hr     Monthly Cost
--------------------------------------------------------------------------------
No batching               5           100         $350.00     $252,000.00
Static (batch=8)         30            17          $59.50      $42,840.00
Continuous (vLLM)        60             9          $31.50      $22,680.00

Savings (no batching -> continuous): $229,320.00/month (91.0%)


---
# Part 4: Putting It All Together (~10 min)

Now let's see how the three levers (token reduction, model selection, batching) combine for a real scenario.

## 4.1 Cost Optimization Scenario

**Scenario:** A customer support chatbot handles **10,000 conversations/day**.

- Average input: 2,000 tokens, output: 500 tokens per conversation
- Starting model: GPT-4

We apply three optimizations sequentially.

In [21]:
conversations_per_day = 10_000
avg_input_tokens = 2_000
avg_output_tokens = 500

baseline_cost_per_conv = calculate_cost(avg_input_tokens, avg_output_tokens, "gpt-4")
baseline_daily = baseline_cost_per_conv * conversations_per_day
print(f"Baseline (GPT-4, all traffic): ${baseline_daily:,.2f}/day")

reduced_input = int(avg_input_tokens * 0.6)
opt1_cost = calculate_cost(reduced_input, avg_output_tokens, "gpt-4") * conversations_per_day
print(f"After Opt 1 (40% prompt reduction):   ${opt1_cost:,.2f}/day")

gpt4_share = 0.30
gpt35_share = 0.70
opt2_gpt4_cost = calculate_cost(reduced_input, avg_output_tokens, "gpt-4") * conversations_per_day * gpt4_share
opt2_gpt35_cost = calculate_cost(reduced_input, avg_output_tokens, "gpt-3.5-turbo") * conversations_per_day * gpt35_share
opt2_cost = opt2_gpt4_cost + opt2_gpt35_cost
print(f"After Opt 2 (70% routed to GPT-3.5):  ${opt2_cost:,.2f}/day")

self_hosted_cost_per_1k = cost_per_1k_tokens(2.00, 80)
self_hosted_tokens = (reduced_input + avg_output_tokens) * conversations_per_day * gpt35_share
opt3_self_hosted = (self_hosted_tokens / 1000) * self_hosted_cost_per_1k
opt3_cost = opt2_gpt4_cost + opt3_self_hosted
print(f"After Opt 3 (self-hosted + batching):  ${opt3_cost:,.2f}/day")

print(f"\nTotal savings: ${baseline_daily - opt3_cost:,.2f}/day "
      f"({(1 - opt3_cost/baseline_daily)*100:.1f}% reduction)")
print(f"\n=== Monthly Summary ===")
print(f"Baseline:  ${baseline_daily * 30:,.2f}/month")
print(f"Optimized: ${opt3_cost * 30:,.2f}/month")
print(f"Savings:   ${(baseline_daily - opt3_cost) * 30:,.2f}/month")

Baseline (GPT-4, all traffic): $900.00/day
After Opt 1 (40% prompt reduction):   $660.00/day
After Opt 2 (70% routed to GPT-3.5):  $205.80/day
After Opt 3 (self-hosted + batching):  $203.75/day

Total savings: $696.25/day (77.4% reduction)

=== Monthly Summary ===
Baseline:  $27,000.00/month
Optimized: $6,112.50/month
Savings:   $20,887.50/month


In [22]:
stages = ["Baseline\n(GPT-4)",
          "Opt 1:\nPrompt\nReduction",
          "Opt 2:\nModel\nRouting",
          "Opt 3:\nSelf-Hosted\n+ Batching"]
costs = [baseline_daily, opt1_cost, opt2_cost, opt3_cost]

fig, ax = plt.subplots(figsize=(10, 6))

bar_colors = ["#EA4335", "#FBBC04", "#4285F4", "#34A853"]
bars = ax.bar(stages, costs, color=bar_colors, edgecolor="black", width=0.5)

for bar, c in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f"${c:,.0f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

for i in range(1, len(costs)):
    reduction = costs[i-1] - costs[i]
    pct = (reduction / costs[i-1]) * 100
    mid_x = i - 0.5
    mid_y = (costs[i-1] + costs[i]) / 2
    ax.annotate(f"-${reduction:,.0f}\n(-{pct:.0f}%)",
                xy=(mid_x, mid_y), fontsize=9, ha="center",
                color="darkred", fontweight="bold")

ax.set_ylabel("Daily Cost (USD)")
ax.set_title("Cost Optimization Waterfall \u2014 Customer Support Chatbot")
plt.tight_layout()
plt.show()

<Figure: Waterfall Bar Chart — Cost Optimization showing baseline $900/day (red) dropping to $660 after prompt reduction (yellow), $206 after model routing (blue), and $204 after self-hosting (green), with reduction percentages annotated between each step>

## 4.2 The Three-Lever Framework — Quick Reference

In [23]:
print("="*74)
print("         THE THREE-LEVER FRAMEWORK FOR LLM COST OPTIMIZATION")
print("="*74)
print()
print("LEVER 1: TOKENS")
print("  What it controls: How many tokens you send and receive")
print("  Key techniques:   Prompt compression, output constraints,")
print("                    summarization, shorter system prompts,")
print("                    caching repeated prompts")
print("  Typical savings:  30-60%")
print()
print("LEVER 2: MODEL / HARDWARE")
print("  What it controls: Cost per token")
print("  Key techniques:   Model routing (cheap model for easy tasks),")
print("                    quantization (INT8/INT4), GPU selection,")
print("                    self-hosting at scale")
print("  Typical savings:  50-90%")
print()
print("LEVER 3: BATCHING")
print("  What it controls: Throughput and GPU utilization")
print("  Key techniques:   Continuous batching (vLLM/TGI),")
print("                    PagedAttention, dynamic batch sizing")
print("  Typical savings:  2-5x throughput (50-80% cost reduction)")
print()
print("COMBINED: 80-95% total cost reduction is achievable")
print("="*74)

         THE THREE-LEVER FRAMEWORK FOR LLM COST OPTIMIZATION

LEVER 1: TOKENS
  What it controls: How many tokens you send and receive
  Key techniques:   Prompt compression, output constraints,
                    summarization, shorter system prompts,
                    caching repeated prompts
  Typical savings:  30-60%

LEVER 2: MODEL / HARDWARE
  What it controls: Cost per token
  Key techniques:   Model routing (cheap model for easy tasks),
                    quantization (INT8/INT4), GPU selection,
                    self-hosting at scale
  Typical savings:  50-90%

LEVER 3: BATCHING
  What it controls: Throughput and GPU utilization
  Key techniques:   Continuous batching (vLLM/TGI),
                    PagedAttention, dynamic batch sizing
  Typical savings:  2-5x throughput (50-80% cost reduction)

COMBINED: 80-95% total cost reduction is achievable


---
# Summary

## The Three-Lever Framework for LLM Cost Optimization

| Lever | What It Controls | Key Techniques |
|-------|-----------------|----------------|
| **1. Tokens** | How many tokens you send and receive | Prompt compression, summarization, caching, shorter system prompts |
| **2. Model / Hardware** | Cost per token | Model routing (cheap model for easy tasks), quantization (INT8/INT4), GPU selection |
| **3. Batching** | Throughput and utilization | Static vs continuous batching, dynamic batch sizing, request scheduling |

### Key Takeaways

1. **Tokenizer choice matters** -- the same text produces different token counts across models, directly affecting cost.

2. **Quantization unlocks cheaper hardware** -- INT8 and INT4 let you fit larger models on smaller (cheaper) GPUs with modest quality trade-offs.

3. **Continuous batching dramatically improves utilization** -- replacing completed requests immediately keeps the GPU busy and reduces per-request cost.

4. **Combining all three levers is multiplicative** -- a 40% token reduction + 70% model routing + self-hosted batching can yield 80%+ cost savings.

5. **Always measure the quality impact** -- cost savings mean nothing if output quality drops below acceptable thresholds.

---
*Cost & Performance Trade-offs -- Masai School | IITP-AIML-2506 | Week 34, Session 2*